# Quickstart: SurvBART for Causal Survival Analysis

This tutorial demonstrates the modern `SurvBART` API for discrete-time survival analysis using Bayesian Additive Regression Trees (BART).

### Key Capabilities Demonstrated:
1. **Synthetic Clinical Cohort Generation**: `simulate_confounded_survival_data` with known ground-truth treatment hazard ratio (zero PHI).
2. **Convenience Workflow**: Direct tabular fitting via `model.fit_survival()`.
3. **Cumulative Survival Prediction**: Out-of-sample $S(t)$ curves with 95% Credible Intervals.
4. **Variable Inclusion Feature Importance**: Normalized split proportions across the posterior tree ensemble.
5. **Support-Audited Causal G-Computation**: Automated detection and donor resampling for conditional support / positivity violations (e.g. inpatient vs. length of stay), plus continuous shift interventions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bart_survival import SurvBART, simulate_confounded_survival_data

# Set plotting style
plt.style.use("seaborn-v0_8-whitegrid") if "seaborn-v0_8-whitegrid" in plt.style.available else None

## 1. Simulate Observational Clinical Cohort
We generate synthetic survival data with an observed treatment hazard ratio of 1.75 and a conditional support constraint (outpatients have `los = 0.0`).

In [ ]:
true_hr = 1.75
df, meta = simulate_confounded_survival_data(
    N=350,
    treatment_hr=true_hr,
    censoring_rate=0.25,
    seed=42
)
feature_cols = ["inpatient", "age", "stage", "los"]
X = df[feature_cols]

print(f"Cohort size: {len(df)} subjects")
print(f"Observed event rate: {df['status'].mean():.1%}")
print(f"Median follow-up time: {df['time'].median():.1f} days")
df.head()

## 2. Initialize and Fit SurvBART
We initialize `SurvBART` using `preset_standard()` (50 trees, 1000 draws, 250 tune, 4 chains, thin=4) and fit directly from tabular covariates.

In [ ]:
# Initialize model with standard scientific preset
model = SurvBART.preset_standard()

# Direct tabular fit
model.fit_survival(
    X=X,
    time=df["time"],
    status=df["status"],
    time_scale=5.0,  # 5-day discrete intervals
    progressbar=True,
    random_seed=42
)
print(f"Fitted successfully across {len(model.training_times_)} discrete time intervals.")

## 3. Out-of-Sample Survival Curves S(t)
We predict cumulative survival curves with 95% Credible Intervals for the cohort.

In [ ]:
surv_res = model.predict_survival(X, return_quantiles=True)
times = surv_res["times"]
mean_st = surv_res["mean"].mean(axis=0)
lower_st = surv_res["lower"].mean(axis=0)
upper_st = surv_res["upper"].mean(axis=0)

plt.figure(figsize=(8, 5))
plt.step(times * 5.0, mean_st, label="Posterior Mean S(t)", color="darkblue", lw=2, where="post")
plt.fill_between(times * 5.0, lower_st, upper_st, step="post", color="cornflowerblue", alpha=0.3, label="95% Credible Interval")
plt.xlabel("Follow-up Time (Days)", fontsize=12)
plt.ylabel("Cumulative Survival Probability S(t)", fontsize=12)
plt.title("Population-Average Cumulative Survival Curve", fontsize=14, weight="bold")
plt.ylim(0, 1.05)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

## 4. Posterior Feature Importance
Variable inclusion proportions reflect the percentage of tree branch splits allocated to each predictor.

In [ ]:
feat_imp = model.feature_importance(sort=True)
print(feat_imp)

plt.figure(figsize=(7, 4))
feat_imp.plot(kind="barh", color="teal")
plt.xlabel("Relative Variable Inclusion Proportion", fontsize=12)
plt.title("Posterior Feature Importance", fontsize=14, weight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Support-Audited Causal Treatment Effect Estimation & Shift Interventions
Causal G-computation audits conditional support for structural positivity violations (e.g., zero LOS for outpatients) and re-routes dependent features to conditional donor resampling.

In [ ]:
causal_effect = model.estimate_effect(
    X=X,
    primary_col="inpatient",
    values=[0, 1],
    audit_support=True,
    donor_seed=42
)

weighted_hr = causal_effect["hazard_ratio_weighted_mean"]
weighted_ci = causal_effect["hazard_ratio_weighted_ci"]
rd_horizon = causal_effect["risk_difference_horizon_mean"]

print(f"Flagged support violations handled: {causal_effect['flagged_support_violations']}")
print(f"Ground-truth Hazard Ratio:  {true_hr:.2f}")
print(f"Event-Weighted Summary HR:  {weighted_hr:.2f} (95% CI: [{weighted_ci[0]:.2f}, {weighted_ci[1]:.2f}])")
print(f"Horizon Risk Difference:    {rd_horizon:+.2%} (Excess risk: Arm 1 - Arm 0)")

# Continuous Shift Intervention (+5 years of age)
shift_effect = model.estimate_effect(X=X, primary_col="age", delta=5.0)
print(f"\nAge (+5 yrs) Shift HR:      {shift_effect['hazard_ratio_weighted_mean']:.2f} (95% CI: [{shift_effect['hazard_ratio_weighted_ci'][0]:.2f}, {shift_effect['hazard_ratio_weighted_ci'][1]:.2f}])")
print(f"Age (+5 yrs) Horizon RD:    {shift_effect['risk_difference_horizon_mean']:+.2%}")